# Two Stage Retrieval RAG Exploration

In [1]:
import ast
import pandas as pd
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from dotenv import load_dotenv
import os

/Users/muhammadabdelmohsen/Desktop/PersonalGrowth/ITI/NLP/NLP/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()

True

# Embedding Model Initialization

In [3]:
DS_NAME = "../../RAG/grouped_mental_health_counseling_conversations_combined_dataset.csv"

In [4]:
DS_NAME

'../../RAG/grouped_mental_health_counseling_conversations_combined_dataset.csv'

In [3]:
device = os.getenv("DEVICE")
device

'cpu'

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name=os.getenv("EMBEDDING_MODEL"),
    model_kwargs={"device": os.getenv("DEVICE")},
    encode_kwargs={"normalize_embeddings": True},
)

2026-05-29 23:21:22.951779: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
client = QdrantClient(path="./qdrant_db")
# client.create_collection(
#     collection_name="mental_health",
#     vectors_config=VectorParams(
#         size=int(os.getenv("EMBEDDING_SIZE")), distance=Distance.COSINE
#     ),
# )

In [9]:
df = pd.read_csv(DS_NAME)
df.head(20)

,Unnamed: 0,Context,Response
0,0,A few nights ago I talked to this girl I know ...,['Hey! \xa0It takes a lot of courage to share ...
1,1,A few nights ago I talked to this girl I know ...,['Hey! \xa0It takes a lot of courage to share ...
2,2,A few years ago I was making love to my wife w...,"[""First step always is to do a medical rule ou..."
3,3,A few years ago I was making love to my wife w...,"[""First step always is to do a medical rule ou..."
4,4,A friend of mine taking psychology advised I g...,"[""I admire your courage for stating your view ..."
5,5,A girl and I were madly in love. We dated for ...,"['Hi Boise, I\'m sorry that you\'ve lost this ..."
6,6,"A lot of times, I avoid situations where I am ...","['Hello, and thank you for your question. Firs..."
7,7,"A year ago, the love of my life left me and ne...",['I recognize that you say you are missing bei...
8,8,"A year ago, the love of my life left me and ne...","['Who takes care of your son, is a significant..."
9,9,"About 3 years ago or so I was skinny, but I wa...",['Hey! \xa0I am so impressed with your efforts...


In [10]:
df = df.dropna(subset=["Response"])
df

,Unnamed: 0,Context,Response
0,0,A few nights ago I talked to this girl I know ...,['Hey! \xa0It takes a lot of courage to share ...
1,1,A few nights ago I talked to this girl I know ...,['Hey! \xa0It takes a lot of courage to share ...
2,2,A few years ago I was making love to my wife w...,"[""First step always is to do a medical rule ou..."
3,3,A few years ago I was making love to my wife w...,"[""First step always is to do a medical rule ou..."
4,4,A friend of mine taking psychology advised I g...,"[""I admire your courage for stating your view ..."
...,...,...,...
989,989,"Whether it's to a guy or girl, I always feel i...","['Hi. I\'m glad you wrote, because I think a l..."
990,990,Why am I attracted to older men?,['What a wonderful question!Good for you on cl...
991,991,Why am I so afraid of it? I don't understand.,"[""Your fear is somewhat reasonable. \xa0No one..."
992,992,he just walks in the house whenever he wants t...,"['The short answer to your question is ""No"" it..."


In [ ]:
docs = [
    Document(
        page_content=row["Context"],
        metadata={"Response": ast.literal_eval(row["Response"])},
    )
    for _, row in df.iterrows()
]

In [ ]:
docs[0].page_content

"A few nights ago I talked to this girl I know about my self esteem issues for the first time. We talked for hours and she told me time and again that I was a great guy. She told me I was attractive, and have a great personality, etc. I really started to feel better about myself by the time I woke up the next morning.\nNow, though, I can't stop thinking about her, but I leave to go back to college in a few days and I go to school 4 hours away from her. So now I feel constantly depressed because even if I told her how I felt it wouldn't matter. I feel helpless and I don't know what to do."

In [12]:
vectorstore = QdrantVectorStore(
    client=client,
    collection_name="mental_health",
    embedding=embeddings,
)

# vectorstore.add_documents(docs)
# print(f"inserted {len(docs)} documents")

In [7]:
# load the vectorstore from the existing collection
vectorstore = QdrantVectorStore(
    client=client,
    collection_name="mental_health",
    embedding=embeddings,
)

In [8]:
retrieved_docs = vectorstore.similarity_search("I am feeling very anxious and stressed lately", k=3)

In [9]:
for pagecontent in retrieved_docs:
    print(pagecontent.page_content)

I've been having horrible anxiety for the last week. I can't sleep. I get a sense of doom, and it's hard to breathe. I feel like nothing I do makes it better.
I haven't been feeling like myself lately. I've been upset for no reason and feeling anxious. I'm feeling burnt out. What can help me feel better?
I need help dealing with stress. How can I handle it all and feel less stressed out?


In [10]:
for answer in retrieved_docs[1].metadata["Response"]:
    print(answer)
    print("------")

Does it help to put a name to the experience you are having? Where you first said, "I've been upset for no reason and feeling anxious," you then say that you are "feeling burnt out," which gives a little more context to what may have once felt like "no reason." Perhaps you are feeling burnt out! That is a big deal, and please try not to take it lightly! When we get burnt out, I have found that it's from one of two things: either we are not doing what we want, or we are doing too much (either of something we want or something we don't, doesn't really matter once we get into doing too much.)If either of those rings true for your experience, try as much as possible to sit with the experience and get a better sense of where the burn out is coming from. I wrote about burnout for a newsletter and it is on my website. I don't want to do shameless promotion, but I thought it could also add to helping you: https://davidkleintherapy.com/my-experiences-with-burnout/I hope that you can also see th

In [23]:
# now for the two stage RAG
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("cross-encoder/stsb-roberta-base", device=device)
# cross_encoder = CrossEncoder("BAAI/bge-reranker-base", device=device)

In [29]:
user_query = "I'm feeling so overwhelmed and hopeless tonight, I can't calm down."
retrieved_docs = vectorstore.similarity_search_with_score(user_query, k=10)
# print the similarity scores and the retrieved answers
retrieved_docs

[(Document(metadata={'Response': ['Anxiety is usually a sign of a current problem to which familiar emotional patterns of feeling similarly upset, attach themselves.Try to understand more about who you are, what you like, feel uneasy about, especially your deeper emotions of being emotionally harmed or injured by meaningful people.Anxiety is best addressed indirectly by understanding and kindly accepting previous hurt and fears from long ago.Once you feel at ease with dynamics of past situations then the current anxiety will decrease. \xa0This is because you will have adjusted and found new ways of handling otherwise frightening and overwhelming interactions and involvements with others.'], '_id': 'f2c70f58abbe4795b91e30865a1f7ac7', '_collection_name': 'mental_health'}, page_content="I've been having horrible anxiety for the last week. I can't sleep. I get a sense of doom, and it's hard to breathe. I feel like nothing I do makes it better."),
  0.7383030173233074),
 (Document(metadata=

In [30]:
# ✅ Correct — rerank all retrieved docs by their context
retrieved_answers = []
retrieved_questions = [question.page_content for question, _ in retrieved_docs]  # extract the questions from the retrieved docs
print("Retrieved Questions:")
# print(retrieved_questions)
for doc in retrieved_docs:
    score = doc[1]  # get the similarity score from metadata
    doc = doc[0]  # extract the Document object from the (Document, score) tuple
    # score = doc[1]  # get the similarity score from metadata
    for answer in doc.metadata["Response"]:
        retrieved_answers.append(answer)
        print(f"Similarity Score: {score:.4f} | Answer: {answer[:60]}...")
        
scores = cross_encoder.predict([(user_query, answer) for answer in retrieved_answers])

Retrieved Questions:
Similarity Score: 0.7383 | Answer: Anxiety is usually a sign of a current problem to which fami...
Similarity Score: 0.7015 | Answer: Maybe your thoughts require your attention and the best cour...
Similarity Score: 0.7015 | Answer: I imagine that it's pretty disconcerting to feel as though y...
Similarity Score: 0.6945 | Answer: Good question. There are resources out there - people to tal...
Similarity Score: 0.6907 | Answer: This is a very common question in my practice. Panic attacks...
Similarity Score: 0.6907 | Answer: The other two post answers to your question are very good an...
Similarity Score: 0.6907 | Answer: Anxiety is simply your system communicating to you that you ...
Similarity Score: 0.6907 | Answer: The are two ways that such anxiety can be dealt with. One is...
Similarity Score: 0.6907 | Answer: The other two post answers to your question are very good an...
Similarity Score: 0.6907 | Answer: Anxiety is simply your system communicating to you th

In [31]:
for doc in retrieved_docs:
    print(doc[0].metadata)

{'Response': ['Anxiety is usually a sign of a current problem to which familiar emotional patterns of feeling similarly upset, attach themselves.Try to understand more about who you are, what you like, feel uneasy about, especially your deeper emotions of being emotionally harmed or injured by meaningful people.Anxiety is best addressed indirectly by understanding and kindly accepting previous hurt and fears from long ago.Once you feel at ease with dynamics of past situations then the current anxiety will decrease. \xa0This is because you will have adjusted and found new ways of handling otherwise frightening and overwhelming interactions and involvements with others.'], '_id': 'f2c70f58abbe4795b91e30865a1f7ac7', '_collection_name': 'mental_health'}
{'Response': ["Maybe your thoughts require your attention and the best course would be to pay attention and follow them!If you're in an especially stressful or uncertain time in your life, then the best way through is to understand the tens

In [32]:
scores

array([0.41494784, 0.37341365, 0.4153701 , 0.14920434, 0.49419254,
       0.5764237 , 0.43643168, 0.6228509 , 0.38081592, 0.42446595,
       0.5250434 , 0.29419768, 0.52677274, 0.24036534, 0.24621113,
       0.51403445, 0.51731247, 0.40010282, 0.44126168], dtype=float32)

In [33]:
ranked_results = sorted(zip(scores, retrieved_answers), key=lambda x: x[0], reverse=True)

# Print the results
for score, answer in ranked_results:
    print(col := f"Score: {score:.4f} | Answer: {answer[:60]}...")

Score: 0.6229 | Answer: The are two ways that such anxiety can be dealt with. One is...
Score: 0.5764 | Answer: The other two post answers to your question are very good an...
Score: 0.5268 | Answer: One approach is to be more accepting of yourself as someone ...
Score: 0.5250 | Answer: It sounds like you are feeling like things are hopeless and ...
Score: 0.5173 | Answer: There's no such possibility that you're upset for "no reason...
Score: 0.5140 | Answer: Does it help to put a name to the experience you are having?...
Score: 0.4942 | Answer: This is a very common question in my practice. Panic attacks...
Score: 0.4413 | Answer: I'm sorry that you have tried several different things and n...
Score: 0.4364 | Answer: Anxiety is simply your system communicating to you that you ...
Score: 0.4245 | Answer: Anxiety is simply your system communicating to you that you ...
Score: 0.4154 | Answer: I imagine that it's pretty disconcerting to feel as though y...
Score: 0.4149 | Answer: Anxiety 